Import dependencies 

In [79]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import joblib
import os
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report,confusion_matrix


In [34]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Data Pre-processing

In [35]:
#Import the dataset
real=pd.read_csv("D:\\TruthLens\\Data\\True.csv")
fake=pd.read_csv("D:\\TruthLens\\Data\\Fake.csv")

In [36]:
#Create labels
real["label"] = 0
fake["label"] = 1

In [37]:
real.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0


In [38]:
fake.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [39]:
#Combining the datasets
df=pd.concat([real,fake],ignore_index=True)

In [40]:
df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0


In [41]:
#Shuffle the data
df = df.sample(frac=1, random_state=40).reset_index(drop=True)

In [42]:
df.head()


,title,text,subject,date,label
0,FBI UNDERCOVER Informant On Hillary’s 2010 Sal...,The FBI informant who went undercover to look ...,Government News,"Nov 17, 2017",1
1,BREAKING #BetsyDeVos CONFIRMED: DEMOCRATS SEET...,Watch History in the making!@VP breaks the TIE...,politics,"Feb 7, 2017",1
2,"U.S. plans to admit maximum 45,000 refugees in...",WASHINGTON (Reuters) - The Trump administratio...,politicsNews,"September 27, 2017",0
3,Senate Republican tax chief wants dividend ded...,WASHINGTON (Reuters) - The U.S. Senate’s top R...,politicsNews,"September 19, 2017",0
4,Victims Of Terrorist Attack Question Lack Of ...,Minnesota s governor did something the current...,News,"August 8, 2017",1


In [43]:
df.shape

(44898, 5)

In [44]:
df.columns

Index(['title', 'text', 'subject', 'date', 'label'], dtype='str')

In [45]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [46]:
df.tail()

,title,text,subject,date,label
44893,UK PM May says she was unaware of allegations ...,LONDON (Reuters) - British Prime Minister Ther...,worldnews,"December 22, 2017",0
44894,Oregon Won’t Prosecute Baby’s Abuser Because ...,An anguished Oregon couple took to social medi...,News,"May 22, 2016",1
44895,Lebanon FM says Hariri crisis an attempt to cr...,BEIRUT (Reuters) - Lebanon s Foreign Minister ...,worldnews,"November 17, 2017",0
44896,JUST IN: Flynn to Plead Guilty to Lying to the...,The broader story in this entire Russia witch ...,politics,"Dec 1, 2017",1
44897,"Israel rejects U.N. vote, thanks Trump for sta...",JERUSALEM (Reuters) - Israel rejected a U.N. v...,worldnews,"December 21, 2017",0


In [47]:
df["label"].value_counts()

label
1    23481
0    21417
Name: count, dtype: int64

In [48]:
#Combining the title and text columns into a single content column
df['content']=df['title']+' '+df['text']

In [49]:
print(df['content'])

0        FBI UNDERCOVER Informant On Hillary’s 2010 Sal...
1        BREAKING #BetsyDeVos CONFIRMED: DEMOCRATS SEET...
2        U.S. plans to admit maximum 45,000 refugees in...
3        Senate Republican tax chief wants dividend ded...
4         Victims Of Terrorist Attack Question Lack Of ...
                               ...                        
44893    UK PM May says she was unaware of allegations ...
44894     Oregon Won’t Prosecute Baby’s Abuser Because ...
44895    Lebanon FM says Hariri crisis an attempt to cr...
44896    JUST IN: Flynn to Plead Guilty to Lying to the...
44897    Israel rejects U.N. vote, thanks Trump for sta...
Name: content, Length: 44898, dtype: str


In [50]:
df.to_csv("D:\\TruthLens\\Data\\combined_data.csv", index=False)

Stemming

In [51]:
#Loading PorterStemming function
port_stem=PorterStemmer()

In [54]:
#Loading stopwords
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

In [55]:
#Creating a function for stemming the content
def stemming(content):
    stemmed_content=re.sub('[^a-zA-Z]',' ',content)
    stemmed_content=stemmed_content.lower()
    stemmed_content=stemmed_content.split()
    stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stop_words]
    stemmed_content=' '.join(stemmed_content)
    return stemmed_content

In [56]:
#Applying the stemming function to the content column
df['content']=df['content'].apply(stemming)

In [57]:
print(df['content'])

0        fbi undercov inform hillari sale uranium ident...
1        break betsydevo confirm democrat seeth champio...
2        u plan admit maximum refuge next fiscal year w...
3        senat republican tax chief want dividend deduc...
4        victim terrorist attack question lack presiden...
                               ...                        
44893    uk pm may say unawar alleg sack deputi london ...
44894    oregon prosecut babi abus young testifi anguis...
44895    lebanon fm say hariri crisi attempt creat regi...
44896    flynn plead guilti lie fbi broader stori entir...
44897    israel reject u n vote thank trump stanc jerus...
Name: content, Length: 44898, dtype: str


In [58]:
#Separating the data & label
X=df['content'].values
Y=df['label'].values

In [59]:
print(X)

<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [60]:
print(Y)

[1 1 0 ... 0 1 0]


Converting Textual data into Numerical data

In [61]:
vectorizer=TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

In [62]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6904081 stored elements and shape (44898, 89868)>
  Coords	Values
  (0, 298)	0.023747845036935472
  (0, 494)	0.01685224017805291
  (0, 619)	0.020840417796011493
  (0, 821)	0.03677719969928287
  (0, 1285)	0.025450499208534795
  (0, 2217)	0.025271402861005852
  (0, 2301)	0.01275067482661736
  (0, 2562)	0.01823773546464223
  (0, 3154)	0.021371249943336477
  (0, 3186)	0.03694833028616671
  (0, 3598)	0.04937758318370229
  (0, 4617)	0.04328177276557116
  (0, 4688)	0.09886494736494873
  (0, 5357)	0.015376973250360708
  (0, 7020)	0.037273793156343375
  (0, 7298)	0.02379386174510396
  (0, 7670)	0.020536401448624265
  (0, 8254)	0.05190845888438443
  (0, 8564)	0.02795566593442498
  (0, 9031)	0.034783445091770324
  (0, 9051)	0.03738207913190706
  (0, 9445)	0.036920567083691694
  (0, 9670)	0.08722983594913031
  (0, 10418)	0.031572433862100825
  (0, 10537)	0.08207251881362854
  :	:
  (44897, 49701)	0.06290623443044062
  (44897, 53138)	0.1

Splitting the dataset to training and test data

In [63]:
X_train, X_test, Y_train, Y_test=train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=40)

Training the model

In [64]:
svm_model = LinearSVC(C=1.0, random_state=42)

svm_model.fit(X_train, Y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adj

In [65]:
Y_pred = svm_model.predict(X_test)

Evaluating model

In [70]:
accuracy = accuracy_score(Y_test, Y_pred)
precision = precision_score(Y_test, Y_pred, pos_label=1)
recall = recall_score(Y_test, Y_pred, pos_label=1)
f1 = f1_score(Y_test, Y_pred, pos_label=1)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

Accuracy : 0.994097995545657
Precision: 0.9948838200810062
Recall   : 0.993824531516184
F1 Score : 0.9943538936827527

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4284
           1       0.99      0.99      0.99      4696

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [75]:
#Confusion Matrix
cm = confusion_matrix(Y_test, Y_pred)
print(cm)

[[4260   24]
 [  29 4667]]


In [77]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = svm_model.coef_[0]

top_fake = feature_names[np.argsort(coefficients)[:20]]
top_real = feature_names[np.argsort(coefficients)[-20:][::-1]]

print("Top FAKE-related features:")
print(top_fake)

print("\nTop REAL-related features:")
print(top_real)

Top FAKE-related features:
['reuter' 'said' 'nov' 'washington' 'wednesday' 'factbox' 'thursday'
 'newspap' 'tuesday' 'rival' 'barack' 'friday' 'edt' 'km' 'urg' 'monday'
 'say' 'presidenti' 'comment' 'york']

Top REAL-related features:
['via' 'read' 'video' 'imag' 'gop' 'us' 'hillari' 'sen' 'getti' 'featur'
 'break' 'mr' 'wire' 'rep' 'com' 'america' 'entir' 'daili' 'even' 'watch']


In [78]:
decision_scores = svm_model.decision_function(X_test)

print(decision_scores[:10])

[-1.5672823   1.32844152 -0.96125561  2.33141485  0.19300485  1.28537084
  1.54567071  2.49596494  1.7977482   1.47846743]


Saving the model

In [82]:
joblib.dump(vectorizer, "D:\\TruthLens\\Model\\tfidf_vectorizer.pkl")
joblib.dump(svm_model, "D:\\TruthLens\\Model\\svm_model.pkl")

print("Model saved successfully!")

Model saved successfully!
